# Feature Engineering: Ratings Dataset (Pandas)

This notebook performs data quality checks and feature engineering on the `ratings.parquet` dataset using **Pandas**.

## Schema (Input)
- **work_key** (String, nullable)
- **edition_key** (String, nullable)
- **rating** (Int8, nullable)
- **rating_date** (Date32, nullable)

## Objectives
1. Assess data quality (null values, rating range, date validity)
2. Validate `work_key` (required for joins; must start with `/works/`)
3. Validate `rating` values (typical range 1–5)
4. Validate `rating_date` (remove future dates; flag very old)
5. Derive `rating_year` from `rating_date` for temporal analysis
6. Generate quality report

In [27]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuration
DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

## Step 1: Load and Inspect Raw Data

In [28]:
# Load ratings data
input_path = PROCESSED_DIR / 'ratings.parquet'
df = pd.read_parquet(input_path)

print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSchema:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Total rows: 590,986
Columns: ['work_key', 'edition_key', 'rating', 'rating_date']

Schema:
work_key          str
edition_key       str
rating           int8
rating_date    object
dtype: object

Memory usage: 50.84 MB


In [29]:
# Display first few rows
print(df.head(10))
print(df.tail(10))

             work_key         edition_key  rating rating_date
0  /works/OL17882343W                           3  2018-06-20
1   /works/OL1629179W  /books/OL22981670M       5  2018-06-20
2   /works/OL4226036W  /books/OL10690412M       5  2018-06-20
3   /works/OL5264255W   /books/OL2719185M       5  2018-06-20
4   /works/OL1681415W   /books/OL2582724M       5  2018-06-20
5   /works/OL1837390W                           5  2018-06-20
6   /works/OL1853596W   /books/OL2331075M       2  2018-06-20
7   /works/OL9468629W  /books/OL10694710M       5  2022-09-27
8   /works/OL3270902W                           4  2018-06-20
9    /works/OL465360W  /books/OL26427670M       5  2018-06-20
                  work_key         edition_key  rating rating_date
590976   /works/OL9740943W  /books/OL11759177M       4  2022-07-30
590977  /works/OL17419741W  /books/OL26462754M       4  2022-07-30
590978  /works/OL17362458W  /books/OL10691113M       5  2022-07-30
590979  /works/OL15692710W  /books/OL24620998M    

## Step 2: Data Quality Assessment

In [30]:
# Check null values
print("=== Null Value Counts ===")
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df)) * 100

quality_df = pd.DataFrame({
    'Column': null_counts.index,
    'Null Count': null_counts.values,
    'Null Percentage': null_pct.values
})
print(quality_df.to_string(index=False))

=== Null Value Counts ===
     Column  Null Count  Null Percentage
   work_key           0              0.0
edition_key           0              0.0
     rating           0              0.0
rating_date           0              0.0


In [31]:
# Rating value distribution
print("=== Rating Value Distribution ===")
rating_non_null = df['rating'].dropna()
if len(rating_non_null) > 0:
    print(rating_non_null.value_counts().sort_index())
    print(f"\nMin: {rating_non_null.min()}, Max: {rating_non_null.max()}")
    print(f"Mean: {rating_non_null.mean():.2f}, Median: {rating_non_null.median():.0f}")

=== Rating Value Distribution ===
rating
1     31341
2     33217
3    103135
4    160204
5    263089
Name: count, dtype: int64

Min: 1, Max: 5
Mean: 4.00, Median: 4


In [32]:
# rating_date analysis
print("=== rating_date Analysis ===")
dates_non_null = df['rating_date'].dropna()
if len(dates_non_null) > 0:
    print(f"Non-null rating_date count: {len(dates_non_null):,}")
    print(f"Min date: {dates_non_null.min()}")
    print(f"Max date: {dates_non_null.max()}")
    # Future dates (after today) - use .date() so types match Parquet date
    today = pd.Timestamp.now().date()
    future = (dates_non_null > today).sum()
    print(f"Future dates (after today): {future:,}")
    # Very old (before 1900)
    old_cutoff = pd.Timestamp('1900-01-01').date()
    very_old = (dates_non_null < old_cutoff).sum()
    print(f"Dates before 1900: {very_old:,}")

=== rating_date Analysis ===
Non-null rating_date count: 590,986
Min date: 2018-05-05
Max date: 2025-12-31
Future dates (after today): 0
Dates before 1900: 0


In [33]:
# work_key validation: required for joins, should start with /works/
print("=== work_key Validation ===")
work_key_non_null = df['work_key'].dropna()
valid_work_key = work_key_non_null.str.startswith('/works/', na=False)
print(f"Rows with non-null work_key: {len(work_key_non_null):,}")
print(f"Rows with work_key starting with /works/: {valid_work_key.sum():,}")
print(f"Rows with null or malformed work_key: {len(df) - valid_work_key.sum():,}")

malformed = work_key_non_null[~valid_work_key]
if len(malformed) > 0:
    print(f"\nSample malformed work_key values:")
    for val in malformed.head(5):
        print(f"  '{val}'")

=== work_key Validation ===
Rows with non-null work_key: 590,986
Rows with work_key starting with /works/: 590,986
Rows with null or malformed work_key: 0


## Step 3: Clean and Transform Data

In [34]:
df_cleaned = df.copy()
print(f"Original row count: {len(df_cleaned):,}")

Original row count: 590,986


In [35]:
# Keep only rows with valid work_key (required for joins)
before_filter = len(df_cleaned)
df_cleaned = df_cleaned[
    df_cleaned['work_key'].notna() &
    (df_cleaned['work_key'].astype(str).str.strip() != '') &
    df_cleaned['work_key'].astype(str).str.startswith('/works/', na=False)
].copy()
rows_removed = before_filter - len(df_cleaned)
print(f"Rows removed (null/invalid work_key): {rows_removed:,}")
print(f"Rows retained: {len(df_cleaned):,}")

Rows removed (null/invalid work_key): 0
Rows retained: 590,986


In [36]:
# Derive rating_year from rating_date (for temporal analysis)
print("Extracting rating_year from rating_date...")
df_cleaned['rating_year'] = pd.to_datetime(df_cleaned['rating_date'], errors='coerce').dt.year
df_cleaned['rating_year'] = df_cleaned['rating_year'].astype('Int16')
valid_year = df_cleaned['rating_year'].notna().sum()
print(f"Rows with valid rating_year: {valid_year:,}")
print(f"Rows with null rating_year: {df_cleaned['rating_year'].isna().sum():,}")

Extracting rating_year from rating_date...
Rows with valid rating_year: 590,986
Rows with null rating_year: 0


In [37]:
# Remove future rating_date (optional: keep for analysis but flag in report)
today = pd.Timestamp.now().date()  # use .date() to match Parquet date type
before_future = len(df_cleaned)
df_cleaned = df_cleaned[
    df_cleaned['rating_date'].isna() | (df_cleaned['rating_date'] <= today)
].copy()
future_removed = before_future - len(df_cleaned)
print(f"Rows with future rating_date removed: {future_removed:,}")
print(f"Rows retained: {len(df_cleaned):,}")

Rows with future rating_date removed: 0
Rows retained: 590,986


In [38]:
# Optional: validate rating range (e.g. 1-5). Adjust min/max if your data uses different scale.
# Only drop if rating is outside a reasonable range (e.g. 1-5); keep null ratings.
RATING_MIN, RATING_MAX = 1, 5
rating_valid = df_cleaned['rating'].isna() | (
    (df_cleaned['rating'] >= RATING_MIN) & (df_cleaned['rating'] <= RATING_MAX)
)
outliers = (~rating_valid).sum()
if outliers > 0:
    print(f"Rows with rating outside [{RATING_MIN}, {RATING_MAX}]: {outliers:,}")
    df_cleaned = df_cleaned[rating_valid].copy()
    print(f"Rows after removing rating outliers: {len(df_cleaned):,}")
else:
    print(f"All ratings within [{RATING_MIN}, {RATING_MAX}] or null. No rows removed.")

All ratings within [1, 5] or null. No rows removed.


In [39]:
# Final columns: work_key, edition_key, rating, rating_date, rating_year
df_cleaned = df_cleaned[["work_key", "edition_key", "rating", "rating_date", "rating_year"]].copy()

print("Final schema:")
print(df_cleaned.dtypes)
print(f"\nFinal row count: {len(df_cleaned):,}")

Final schema:
work_key          str
edition_key       str
rating           int8
rating_date    object
rating_year     Int16
dtype: object

Final row count: 590,986


## Step 4: Final Quality Check

In [40]:
print("=== Final Quality Check ===")
valid_rating_year = df_cleaned['rating_year'].notna().sum()
pct_valid = (valid_rating_year / len(df_cleaned)) * 100 if len(df_cleaned) > 0 else 0
print(f"Rows with valid rating_year: {valid_rating_year:,} ({pct_valid:.2f}%)")
print(f"Rows with null rating_year: {len(df_cleaned) - valid_rating_year:,}")
if valid_rating_year > 0:
    print(f"\nrating_year - Min: {df_cleaned['rating_year'].min()}, Max: {df_cleaned['rating_year'].max()}")
print(f"\nrating (non-null) - Min: {df_cleaned['rating'].min()}, Max: {df_cleaned['rating'].max()}")

=== Final Quality Check ===
Rows with valid rating_year: 590,986 (100.00%)
Rows with null rating_year: 0

rating_year - Min: 2018, Max: 2025

rating (non-null) - Min: 1, Max: 5


In [41]:
print("\nSample of cleaned data:")
df_cleaned.head(20)


Sample of cleaned data:


,work_key,edition_key,rating,rating_date,rating_year
0,/works/OL17882343W,,3,2018-06-20,2018
1,/works/OL1629179W,/books/OL22981670M,5,2018-06-20,2018
2,/works/OL4226036W,/books/OL10690412M,5,2018-06-20,2018
3,/works/OL5264255W,/books/OL2719185M,5,2018-06-20,2018
4,/works/OL1681415W,/books/OL2582724M,5,2018-06-20,2018
5,/works/OL1837390W,,5,2018-06-20,2018
6,/works/OL1853596W,/books/OL2331075M,2,2018-06-20,2018
7,/works/OL9468629W,/books/OL10694710M,5,2022-09-27,2022
8,/works/OL3270902W,,4,2018-06-20,2018
9,/works/OL465360W,/books/OL26427670M,5,2018-06-20,2018


## Step 5: Save Cleaned Data

In [42]:
output_path = PROCESSED_DIR / 'ratings_cleaned.parquet'
df_cleaned.to_parquet(output_path, index=False)
print(f"Saved cleaned data to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024**2:.2f} MB")

Saved cleaned data to ../data/processed/ratings_cleaned.parquet
File size: 9.44 MB


## Step 6: Generate Quality Report

In [43]:
report_path = REPORTS_DIR / 'data_quality_ratings_pandas.md'
valid_count = df_cleaned['rating_year'].notna().sum()
null_year_count = df_cleaned['rating_year'].isna().sum()

report = f"""# Data Quality Report: Ratings Dataset (Pandas)

## Summary
- **Original row count**: {len(df):,}
- **Cleaned row count**: {len(df_cleaned):,}
- **Rows removed**: {len(df) - len(df_cleaned):,} ({(len(df) - len(df_cleaned))/len(df)*100:.2f}%)
- **Rows retained**: {len(df_cleaned)/len(df)*100:.2f}%

## Data Quality Metrics

### Rating Year (derived from rating_date)
- **Rows with valid rating_year**: {valid_count:,} ({valid_count/len(df_cleaned)*100:.2f}%)
- **Rows with null rating_year**: {null_year_count:,} ({null_year_count/len(df_cleaned)*100:.2f}%)

"""
if valid_count > 0:
    report += f"""### Rating Year Statistics
- **Minimum year**: {df_cleaned['rating_year'].min()}
- **Maximum year**: {df_cleaned['rating_year'].max()}

"""
report += """## Schema Changes
- **Added**: `rating_year` (Int16, nullable) - derived from rating_date.year

## Cleaning Steps Applied
1. Removed rows with null or invalid work_key (must start with /works/)
2. Derived rating_year from rating_date for temporal analysis
3. Removed rows with future rating_date
4. Optionally removed rows with rating outside 1-5 (if any)
"""
report_path.write_text(report)
print(f"Quality report saved to {report_path}")
print("\n" + report_path.read_text())

Quality report saved to ../reports/data_quality_ratings_pandas.md

# Data Quality Report: Ratings Dataset (Pandas)

## Summary
- **Original row count**: 590,986
- **Cleaned row count**: 590,986
- **Rows removed**: 0 (0.00%)
- **Rows retained**: 100.00%

## Data Quality Metrics

### Rating Year (derived from rating_date)
- **Rows with valid rating_year**: 590,986 (100.00%)
- **Rows with null rating_year**: 0 (0.00%)

### Rating Year Statistics
- **Minimum year**: 2018
- **Maximum year**: 2025

## Schema Changes
- **Added**: `rating_year` (Int16, nullable) - derived from rating_date.year

## Cleaning Steps Applied
1. Removed rows with null or invalid work_key (must start with /works/)
2. Derived rating_year from rating_date for temporal analysis
3. Removed rows with future rating_date
4. Optionally removed rows with rating outside 1-5 (if any)



## Summary

Done: load & inspect → quality assessment → work_key validation → rating_year derivation → date/rating validation → save `ratings_cleaned.parquet` and quality report.